In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
import dynamics
from basilisk_runner import run_rendezvous

/Users/shash/Documents/Projects/SatelliteRendezvous/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%matplotlib inline

In [3]:
SEMI_MAJOR_AXIS = 6.778e6
GRAV = 3.986004418e14

In [ ]:
initial_state = np.array([
    100.0,  # x, meters
    0.0,    # y
    0.0,    # z
    0.0,    # vx, m/s
    0.0,    # vy
    0.0,    # vz
]).reshape(6, 1)

n = dynamics.orbital_rate(SEMI_MAJOR_AXIS, GRAV)
dt = 0.01
T = 1000

t, X, U, X_est, attitude_history, wheel_torque, thruster_torque = run_rendezvous(n, initial_state, dt, T)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

# Handles U with either len(t) or len(t)-1 samples.
t_control = t[:len(U)]

for i, axis in enumerate(("x", "y", "z")):
    axes[0].plot(t, X[:, i], label=axis)
    axes[1].plot(t, X[:, i + 3], label=f"v{axis}")
    axes[2].step(
        t_control, U[:, i],
        where="post", label=f"u{axis}"
    )

axes[0].set(
    title="Chaser Position Relative to Chief",
    ylabel="Position (m)",
)
axes[1].set(
    title="Chaser Relative Velocity",
    ylabel="Velocity (m/s)",
)
axes[2].set(
    title="Chaser Control Input",
    xlabel="Time (s)",
    ylabel="Acceleration (m/s²)",
)

for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)




fig.tight_layout()

In [ ]:
quaternions = attitude_history[:, :4]
angular_velocity = attitude_history[:, 4:7]
wheel_momentum = attitude_history[:, 7:10]

desired_rotation = Rotation.identity()
actual_rotation = Rotation.from_quat(quaternions)

attitude_error = (actual_rotation.inv() * desired_rotation).as_rotvec()

fig, axes = plt.subplots(5, 1, figsize=(10, 15), sharex=True)

for i, axis in enumerate(("x", "y", "z")):
    axes[0].plot(
        t, np.rad2deg(attitude_error[:, i]), label=axis
    )

    axes[1].plot(
        t, np.rad2deg(angular_velocity[:, i]), label=f"ω{axis}"
    )

    axes[2].plot(
        t, wheel_momentum[:, i], label=f"h{axis}"
    )

    axes[3].step(
        t[:len(wheel_torque)],
        wheel_torque[:, i],
        where="post", label=f"τ{axis}",
    )

    axes[4].step(
        t[:len(thruster_torque)],
        thruster_torque[:, i],
        where="post", label=f"τ{axis}",
    )

# Desired angular velocity at orbital alignment.
axes[1].axhline(
    np.rad2deg(n),
    color="black",
    linestyle="--",
    label="Desired ωz",
)

# Assumed wheel momentum limits from the simulation.
for limit in (-0.2, 0.2):
    axes[2].axhline(
        limit, color="black", linestyle="--", alpha=0.5
    )

axes[0].set(
    title="Attitude Error — Rotation Vector",
    ylabel="Error (deg)",
)
axes[1].set(
    title="Body Angular Velocity Relative to Inertial Space",
    ylabel="Angular velocity (deg/s)",
)
axes[2].set(
    title="Reaction Wheel Momentum",
    ylabel="Momentum (N·m·s)",
)
axes[3].set(
    title="Reaction Wheel Torque on Spacecraft",
    ylabel="Torque (N·m)",
)
axes[4].set(
    title="External Thruster Torque",
    xlabel="Time (s)",
    ylabel="Torque (N·m)",
)

for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for i, axis in enumerate(("x", "y", "z")):
    axes[i].plot(t, X[:, i], label="True")
    axes[i].plot(t, X_est[:, i], "--", label="Estimated")
    axes[i].set_ylabel(f"{axis} (m)")
    axes[i].grid(True)
    axes[i].legend()

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection="3d")
ax.plot(*X[:, :3].T, label="Chaser trajectory")
ax.scatter(*X[0, :3], label="Start", color="tab:green")
ax.scatter(*X[-1, :3], label="Finish", color="tab:orange")
ax.scatter(0, 0, 0, label="Chief", marker="*", color="black", s=110)
ax.set(
    title="Chaser Trajectory Relative to the Chief",
    xlabel="Radial (m)",
    ylabel="Along-track (m)",
    zlabel="Cross-track (m)",
)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

relative_position = X[:, :3]
relative_velocity = X[:, 3:]
range_m = np.linalg.norm(relative_position, axis=1)
relative_speed = np.linalg.norm(relative_velocity, axis=1)

closing_speed = np.divide(
    -np.sum(relative_position * relative_velocity, axis=1),
    range_m,
    out=np.full_like(range_m, np.nan),
    where=range_m > 0,
)

output = Path("saved_runs/rendezvous_basilisk.npz")
output.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    output,
    time_s=t,
    true_state=X,
    estimated_state=X_est,
    estimation_error=X_est - X,
    range_m=range_m,
    relative_speed_mps=relative_speed,
    closing_speed_mps=closing_speed,
    control_time_s=t[:-1],
    applied_acceleration_mps2=U,
    attitude_history=attitude_history,
)

print(f"Saved {output.resolve()}")
print(f"Duration: {t[-1]:.2f} s")
print(f"Range: {range_m[0]:.2f} m → {range_m[-1]:.3f} m")
print(f"Final relative speed: {relative_speed[-1]:.4f} m/s")
print(f"Final closing speed: {closing_speed[-1]:+.4f} m/s")